# ReversalLab: End-to-End Decision-Reversal Tutorial

Welcome to the **ReversalLab** interactive tutorial.

In contextual-bandit and recommender systems, deciding whether to deploy a newly proposed candidate policy $\pi_e$ over an existing baseline $\pi_b$ is commonly done using off-policy evaluation (OPE). However, conventional confidence intervals frequently collapse when target populations experience demographic shifts or distribution changes.

Instead of asking only *"What is the expected value under historical data?"*, ReversalLab asks:
> **"What is the smallest admissible change in population demographics or evaluation assumptions that reverses an evidence-based decision to deploy an ML policy?"**

### In this tutorial, you will learn how to:
1. Generate validated, immutable contextual-bandit logged data using `BanditDataset`.
2. Evaluate single policies using standard and advanced estimators (IPS, SNIPS, DM, DR) with `PolicyValueEvaluator`.
3. Perform paired policy advantage estimation $\Delta = V(\pi_e) - V(\pi_b)$ with multi-factor diagnostic gating.
4. Model demographic shifts using a 1-D two-stratum mixture model and verify the mathematical concavity of confidence bounds.
5. Search for the minimum evidence-reversal perturbation distance $d^*$ and compute forward worst-case robust certificates.
6. Handle diagnostic gate failures and understand why ambiguous evidence is classified as `INCONCLUSIVE`.
7. Export and inspect reproducible, schema-versioned audit reports (`SDK_SCHEMA_VERSION = "1.0.0"`).


In [1]:
import json
from pathlib import Path
import numpy as np

import reversallab as rl
from reversallab.data.synthetic import SyntheticBanditEnvironment

print(f"ReversalLab library version:   {rl.__version__}")
print(f"SDK Report contract schema:    v{rl.SDK_SCHEMA_VERSION}")


ReversalLab library version:   0.1.0rc1
SDK Report contract schema:    v1.0.0


---
## Step 1: Ingest and Validate Logged Bandit Data

Every observation in logged contextual bandit interactions consists of a tuple $(x_i, a_i, r_i, p_i)$:
- $x_i \in \mathbb{R}^D$: Context features observed *before* the action was selected.
- $a_i \in \{0, \dots, K-1\}$: Discrete action selected by the logging policy $\pi_0$.
- $r_i \in \mathbb{R}$: Observed scalar reward for the chosen action.
- $p_i = \pi_0(a_i \mid x_i)$: Probability assigned to the chosen action by the logging policy (must be strictly positive, $p_i > 0$).

ReversalLab encapsulates this in `BanditDataset`, enforcing strict array shapes, finiteness, and positivity contracts.


In [2]:
# Initialize synthetic environment with 3 actions and 2 context dimensions
rng = np.random.default_rng(42)
env = SyntheticBanditEnvironment.random(
    n_actions=3,
    context_dim=2,
    rng=rng,
    reward_scale=1.5,
    bias_scale=0.0,
)

# Generate logged dataset under uniform exploration
logging_policy = rl.UniformPolicy(n_actions=env.n_actions)
dataset = rl.BanditDataset.from_synthetic(
    env,
    logging_policy=logging_policy,
    n_samples=2500,
    seed=101,
)

print(f"Dataset: N = {dataset.n_samples} interactions, K = {dataset.n_actions} actions, D = {dataset.context_dim} context features.")
print(f"Context feature matrix: shape {dataset.context.shape}, mean = {np.mean(dataset.context):.3f}")
print(f"Action sample (first 5):  {dataset.action[:5]}")
print(f"Reward sample (first 5):  {np.round(dataset.reward[:5], 3)}")
print(f"Min logging propensity:   {np.min(dataset.propensity):.4f}")


Dataset: N = 2500 interactions, K = 3 actions, D = 2 context features.
Context feature matrix: shape (2500, 2), mean = -0.016
Action sample (first 5):  [0 0 2 2 1]
Reward sample (first 5):  [1. 1. 0. 1. 0.]
Min logging propensity:   0.3333


---
## Step 2: Evaluating Candidate and Baseline Policies

Next, we specify two policies:
1. **Baseline Incumbent Policy ($\pi_b$):** Uniform exploratory policy $\pi_b(a \mid x) = 1/K$.
2. **Candidate Improved Policy ($\pi_e$):** Linear softmax policy targeting true environment reward logits.

We evaluate their expected values using Doubly Robust (DR) estimation with leakage-safe 3-fold cross-fitting.


In [3]:
# Define baseline and candidate policies
baseline_policy = rl.UniformPolicy(n_actions=env.n_actions)
candidate_policy = rl.LinearSoftmaxPolicy(
    weights=env.reward_weights,
    bias=env.reward_bias,
    inverse_temperature=1.5,
)

# Evaluate baseline policy
base_report = rl.PolicyValueEvaluator.evaluate(
    dataset=dataset,
    policy=baseline_policy,
    estimator="dr",
    policy_name="Baseline (pi_b)",
    cross_fitting_folds=3,
    confidence_level=0.95,
)

# Evaluate candidate policy
cand_report = rl.PolicyValueEvaluator.evaluate(
    dataset=dataset,
    policy=candidate_policy,
    estimator="dr",
    policy_name="Candidate (pi_e)",
    cross_fitting_folds=3,
    confidence_level=0.95,
)

print(base_report.summary())
print()
print(cand_report.summary())


Policy Value Estimation (Baseline (pi_b))
  Estimator:        DR
  Value Estimate:   0.4991 (SE: 0.0086)
  95% Wald CI:      [0.4823, 0.5158]
  Min Propensity:   3.33e-01
  ESS Fraction:     100.0%
  Max Weight Share: 0.0%

Policy Value Estimation (Candidate (pi_e))
  Estimator:        DR
  Value Estimate:   0.7480 (SE: 0.0114)
  95% Wald CI:      [0.7257, 0.7702]
  Min Propensity:   3.33e-01
  ESS Fraction:     45.9%
  Max Weight Share: 0.1%


---
## Step 3: Paired Advantage Estimation & Diagnostic Gating

In real-world decision making, you do not deploy a model simply because its point estimate is high. You require that:
1. **Operational Margin ($\delta$):** The candidate must outperform the incumbent by a minimum business margin (e.g., $\delta = +0.02$).
2. **Statistical Significance:** The lower 95% confidence bound of the paired difference $\Delta = V(\pi_e) - V(\pi_b)$ must exceed $\delta$.
3. **Diagnostic Gating:** The logging policy must provide verified support on the actions actively taken by $\pi_e$ and $\pi_b$, without severe importance weight concentration.


In [4]:
margin = 0.02  # Minimum deployment threshold (+2% reward)
deploy_eps = 0.10  # Certified demographic envelope [q0 - 0.10, q0 + 0.10]
partition = rl.StratumPartition(split_dim=0, split_threshold=0.0)

comparison_report = rl.PolicyComparison.evaluate(
    dataset=dataset,
    candidate_policy=candidate_policy,
    baseline_policy=baseline_policy,
    margin=margin,
    deploy_epsilon=deploy_eps,
    max_epsilon=0.40,
    stratum_partition=partition,
    estimator="dr",
    cross_fitting_folds=3,
    confidence_level=0.95,
    seed=42,
)

print(comparison_report.summary())


REVERSALLAB DECISION-REVERSAL AUDIT REPORT (v1.0.0)
Generated:       2026-11-06T12:01:19.698198+00:00
Dataset:         N = 2500 observations, 3 actions, dim = 2
Evaluation:      Estimator = DR (3-fold cross-fitting)
Policy Margin:   delta = 0.0200 (Confidence level: 95%)

1. PAIRED POLICY ADVANTAGE
   Point Estimate (Delta):   +0.2489 (SE: 0.0084)
   95% Confidence Bound:    [+0.2325, +0.2653]
   Baseline Action:          RECOMMEND (Statistically significant superiority: Lower confidence bound (0.2325) strictly exceeds margin (0.0200).)

2. RELIABILITY DIAGNOSTICS
   Gate Passed:              YES
   Min Logging Propensity:   3.33e-01 (Threshold: 1.00e-04)
   Effective Sample Size:    1148.2 (45.9% of N)
   Max Single-Weight Share:  0.1%

3. DECISION-REVERSAL ROBUSTNESS (1-D Context Mixture Model)
   Stratum Partition:        x[0] >= 0.00
   Reversal Distance (d*):   None detected (Robust across bounds)
   Search Status:            NO_REVERSAL_FOUND_WITHIN_BOUNDS (NO_REVERSAL)
   Taxono

---
## Step 4: Two-Stratum Demographic Sensitivity & Concavity

The population is partitioned across a context split into two demographic strata:
- **Stratum 0:** $x_0 \ge 0.0$
- **Stratum 1:** $x_0 < 0.0$

Under target mixture proportion $q \in [0, 1]$, the paired advantage is:
$$\Delta(q) = q \Delta_0 + (1-q) \Delta_1$$

The lower confidence bound is:
$$L_\Delta(q) = \Delta(q) - z_{\alpha/2} \sqrt{\frac{q^2 \sigma_0^2}{n_0} + \frac{(1-q)^2 \sigma_1^2}{n_1}}$$

### Mathematical Concavity Property
Because the standard error function $f(q) = \sqrt{q^2 \frac{\sigma_0^2}{n_0} + (1-q)^2 \frac{\sigma_1^2}{n_1}}$ is strictly convex (or affine in degenerate zero-variance cases), $-z f(q)$ is strictly concave. Therefore, **$L_\Delta(q)$ is concave on $[0, 1]$**.

By the minimum principle for concave functions on compact convex sets, the minimum of $L_\Delta(q)$ across any operational interval $[q_{\min}, q_{\max}]$ is **guaranteed to occur at one of the two boundary endpoints**!


In [5]:
# Verify the stratum-level estimates
from reversallab.evaluation.cross_fitting import cross_fit_reward_model
from reversallab.evaluation.reward_models import StratifiedLinearRewardModel
from reversallab.reversal.paired import evaluate_paired_difference
from reversallab.reversal.strata import create_two_stratum_model

def model_factory():
    return StratifiedLinearRewardModel(n_actions=dataset.n_actions, reward_type="binary")

pred_rewards = cross_fit_reward_model(dataset.data, model_factory, n_folds=3, seed=42)
diff_res = evaluate_paired_difference(
    "dr",
    dataset.data,
    candidate_policy.action_distribution(dataset.context),
    baseline_policy.action_distribution(dataset.context),
    predicted_rewards=pred_rewards,
)
strata_model = create_two_stratum_model(dataset.context, diff_res.diff_scores, partition=partition, target_mixture=0.50)

print(f"Stratum 0 (x0 >= 0): Mean advantage = {strata_model.stratum_0.mean_diff:+.4f}, SE = {strata_model.stratum_0.standard_error:.4f}, N = {strata_model.stratum_0.n_samples}")
print(f"Stratum 1 (x0 < 0):  Mean advantage = {strata_model.stratum_1.mean_diff:+.4f}, SE = {strata_model.stratum_1.standard_error:.4f}, N = {strata_model.stratum_1.n_samples}")

# Inspect lower bound L_Delta(q) across demographic mixture q in [0, 1]
print("\nMixture proportion q  | Advantage Delta(q) | Lower Bound L_Delta(q)")
print("-" * 58)
for q_val in [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]:
    mean_q, var_q, l_q, u_q = strata_model.evaluate_mixture(q_val, confidence_level=0.95)
    marker = " <-- Target Mixture (q0)" if q_val == 0.5 else ""
    print(f"q = {q_val:4.2f}               | {mean_q:+16.4f}   | {l_q:+16.4f}{marker}")


Stratum 0 (x0 >= 0): Mean advantage = +0.2534, SE = 0.0124, N = 1260
Stratum 1 (x0 < 0):  Mean advantage = +0.2443, SE = 0.0113, N = 1240

Mixture proportion q  | Advantage Delta(q) | Lower Bound L_Delta(q)
----------------------------------------------------------
q = 0.00               |          +0.2443   |          +0.2223
q = 0.20               |          +0.2461   |          +0.2278
q = 0.40               |          +0.2480   |          +0.2316
q = 0.50               |          +0.2489   |          +0.2325 <-- Target Mixture (q0)
q = 0.60               |          +0.2498   |          +0.2328
q = 0.80               |          +0.2516   |          +0.2317
q = 1.00               |          +0.2534   |          +0.2292


---
## Step 5: Evidence Reversal Distance ($d^*$) vs. Forward Robust Bounds

ReversalLab evaluates two complementary perspectives on decision robustness:

1. **Inverse Reversal Search (Method D):** Finds the minimum demographic shift distance $d^* = |q^* - q_0|$ that overturns the deployment decision. A larger $d^*$ indicates greater decision resilience.
2. **Forward Worst-Case Robust Mixture (Method F):** Evaluates the worst-case lower bound $L_{\text{worst}} = \min_{q \in \mathcal{Q}_\epsilon} L_\Delta(q)$ across the certified operational envelope $\mathcal{Q}_\epsilon = [q_0 - \epsilon, q_0 + \epsilon]$.

### Distinction: Evidence Reversal vs. True Performance Reversal
- **Statistical Evidence Reversal:** Occurs when the lower confidence bound drops below the margin ($L(q) \le \delta$). The data no longer provides statistically significant evidence to recommend deployment.
- **True Performance Reversal:** Occurs when the candidate's actual expected advantage becomes negative ($\Delta^*(q) < 0$). In this case, the candidate policy actively harms user outcomes compared to the baseline.


In [6]:
search_info = comparison_report.reversal_search
robust_info = comparison_report.robust_mixture

print("Reversal Search Status:     ", search_info["status"])
print("Minimum Reversal Distance:  ", search_info["reversal_distance"])
print("Transition Category:        ", search_info["taxonomy_category"])
print("Certified Forward Bound:    ", f"{robust_info['worst_case_lower_bound']:+.4f} (across Q_0.10 = [0.40, 0.60])")
print("Robust Decision Rule:       ", robust_info["outcome"])


Reversal Search Status:      NO_REVERSAL_FOUND_WITHIN_BOUNDS
Minimum Reversal Distance:   None
Transition Category:         no_reversal
Certified Forward Bound:     +0.2316 (across Q_0.10 = [0.40, 0.60])
Robust Decision Rule:        RECOMMEND


---
## Step 6: Diagnostic Gate Failure & Inconclusive Decisions

When logging propensities have an active deficit ( < p_i < \tau$ on an action where the candidate policy places non-negligible probability mass $\pi_e(a_i \mid x_i) \gg p_i$), importance weights explode, destabilizing the evaluation variance.

**Note:** A small positive propensity  > 0$ does not inherently imply a huge importance weight (if $\pi_e(a_i \mid x_i) = 0$,  = 0$; if $\pi_e(a_i \mid x_i) \approx p_i$,  \approx 1$). This is an operational variance-stability failure, not a proven mathematical support violation (since  > 0$ and the Radon-Nikodym derivative exists). However, under active deficits, finite-sample central limit behavior breaks down, rendering nominal Wald intervals severely undercovering.

Instead of outputting an uncontrolled point estimate or a spurious RECOMMEND, ReversalLab's diagnostic gate **trips** and conservatively defers the decision to INCONCLUSIVE.


In [7]:
# Create an unsafe dataset with severe logging propensity deficits
n_bad = 200
rng_bad = np.random.default_rng(999)
bad_contexts = rng_bad.normal(size=(n_bad, 2))
bad_actions = np.zeros(n_bad, dtype=int)
bad_rewards = rng_bad.normal(size=n_bad)
# Severe propensity deficit below 1e-4 threshold
bad_propensities = np.full(n_bad, 1e-6)

unsafe_dataset = rl.BanditDataset.from_arrays(
    contexts=bad_contexts,
    actions=bad_actions,
    rewards=bad_rewards,
    propensities=bad_propensities,
    n_actions=3,
)

# Run paired evaluation with diagnostic threshold 1e-4
gated_comparison = rl.PolicyComparison.evaluate(
    dataset=unsafe_dataset,
    candidate_policy=candidate_policy,
    baseline_policy=baseline_policy,
    margin=0.01,
    min_propensity_threshold=1e-4,
)

print(f"Diagnostic Gate Passed?  {gated_comparison.diagnostics['gate_passed']}")
print(f"Rejection Reasons:       {gated_comparison.diagnostics['rejection_reasons']}")
print(f"Deployment Action:       {gated_comparison.decision['outcome']}")
print(f"Reason:                  {gated_comparison.decision['reason']}")

assert gated_comparison.decision["outcome"] == "INCONCLUSIVE"
print("\nSuccessfully verified: Unsafe propensity deficit was caught and safely halted deployment!")


Diagnostic Gate Passed?  False
Rejection Reasons:       ['Active propensity deficit: min propensity (1.00e-06) < threshold (1.00e-04) on action actively evaluated by policy.']
Deployment Action:       INCONCLUSIVE
Reason:                  Diagnostic gate failure: Active propensity deficit: min propensity (1.00e-06) < threshold (1.00e-04) on action actively evaluated by policy.

Successfully verified: Unsafe propensity deficit was caught and safely halted deployment!


---
## Step 7: Exporting Machine-Readable JSON Reports

To guarantee auditability and scientific reproducibility, every comparison can be exported to a structured JSON artifact conforming to `SDK_SCHEMA_VERSION = "1.0.0"`.


In [8]:
out_file = Path("runs/tutorial_audit_report.json")
comparison_report.save(out_file)
print(f"Exported audit report to: {out_file}")

with open(out_file, "r", encoding="utf-8") as f:
    audit_data = json.load(f)

print(f"\nJSON Schema Version: {audit_data['schema_version']}")
print(f"Report Timestamp:    {audit_data['timestamp']}")
print(f"Report Top Keys:     {list(audit_data.keys())}")
print(f"Decision Summary:    {audit_data['decision']}")


Exported audit report to: runs\tutorial_audit_report.json

JSON Schema Version: 1.0.0
Report Timestamp:    2026-11-06T12:01:19.698198+00:00
Report Top Keys:     ['schema_version', 'timestamp', 'metadata', 'diagnostics', 'paired_advantage', 'decision', 'reversal_search', 'simultaneous_band', 'robust_mixture', 'assumptions_and_guarantees', 'warnings']
Decision Summary:    {'outcome': 'RECOMMEND', 'margin': 0.02, 'reason': 'Statistically significant superiority: Lower confidence bound (0.2325) strictly exceeds margin (0.0200).', 'is_actionable': True}
